### 1. Initialize Langfuse Tracing

Connect the existing local Langfuse instance to the caching experiment.

**Purpose:**
- Capture cache operations as Langfuse observations.
- Track execution latency.
- Record cache hits and misses.
- Inspect cache performance through the Langfuse dashboard.

We will use Langfuse's current observation-based tracing API.

In [1]:
import os
from dotenv import load_dotenv
from langfuse import Langfuse

load_dotenv(override=True)

langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_LOCAL_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_LOCAL_SECRET_KEY"),
    base_url=os.getenv("LANGFUSE_LOCAL_BASE_URL")
)

print("Langfuse connected:", langfuse.auth_check())

Langfuse connected: True


### 2. Inspect Parent Document Structure

Before implementing document caching, inspect the existing parent store.

Our parent store contains 76,366 records loaded from `parent_documents.jsonl`.

**Purpose:**
- Understand the parent document structure.
- Identify available metadata fields.
- Identify the document content field.
- Ensure Redis serialization preserves the original records.

We will use `parent_id` as the Redis cache identity.

In [2]:
from pathlib import Path
from collections import defaultdict
from langchain_chroma import Chroma

OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


### 2. Load Parent Documents

Load the parent documents from the JSONL file into a Python dictionary.

Each line contains one JSON record. We use `parent_id` as the dictionary key for efficient parent document lookup.

**Purpose:**
- Load all parent documents.
- Create a parent ID lookup.
- Prepare documents for Redis caching.

In [3]:
import json

parent_store = {}

with open(PARENT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            parent_store[record["parent_id"]] = record

print("Total parent documents:", len(parent_store))

sample_parent_id, sample_parent = next(iter(parent_store.items()))

print("\nParent ID:", sample_parent_id)
print("\nAvailable fields:", list(sample_parent.keys()))

print("\nField types:")
for key, value in sample_parent.items():
    print(f"{key}: {type(value).__name__}")

Total parent documents: 76366

Parent ID: LBezF5gB5uF80_MGGY81_OVERVIEW_001

Available fields: ['parent_id', 'scheme_id', 'child_type', 'source_field', 'full_text', 'metadata']

Field types:
parent_id: str
scheme_id: str
child_type: str
source_field: str
full_text: str
metadata: dict


### 4. Implement Redis Parent Document Cache

Create a reusable Redis caching class for parent documents.

**Cache strategy:**
- Use `parent_id` as the unique cache identifier.
- Check Redis before accessing the parent store.
- Cache hit: Return the complete document from Redis.
- Cache miss: Fetch the document from `parent_store` and store it in Redis.
- Apply TTL to expire cached documents automatically.

**Langfuse integration:**

Trace every document lookup and record:
- Cache hit or miss.
- Parent document ID.
- Lookup latency.
- Document retrieval source.

**Why?**

Multiple users may request information from the same government scheme. Document caching avoids repeatedly accessing the parent store for frequently requested documents.

In [4]:
class RedisDocumentCache:
    def __init__(self, redis_client, parent_store, langfuse, ttl=86400):
        self.redis = redis_client
        self.parent_store = parent_store
        self.langfuse = langfuse
        self.ttl = ttl

        self.hits = 0
        self.misses = 0

    def get_document(self, parent_id):
        key = f"seva:document:v1:{parent_id}"

        start = time.perf_counter()

        with self.langfuse.start_as_current_observation(
            as_type="span",
            name="parent_document_cache_lookup",
            input={"parent_id": parent_id}
        ) as span:

            cached = self.redis.get(key)

            if cached is not None:
                self.hits += 1
                document = json.loads(cached)
                cache_status = "hit"

            else:
                self.misses += 1
                document = self.parent_store.get(parent_id)
                cache_status = "miss"

                if document is not None:
                    self.redis.set(
                        key,
                        json.dumps(document),
                        ex=self.ttl
                    )

            latency = (time.perf_counter() - start) * 1000

            span.update(
                output={
                    "cache_status": cache_status,
                    "document_found": document is not None
                },
                metadata={
                    "parent_id": parent_id,
                    "latency_ms": round(latency, 2)
                }
            )

        return document

### 5. Initialize Redis Document Cache

Initialize the document cache using our existing Redis connection, parent document store and Langfuse client.

**Configuration:**
- Redis: Existing Redis 8 instance.
- Parent store: 76,366 parent documents.
- TTL: 86,400 seconds (24 hours).
- Langfuse: Existing local instance.

The cache uses a versioned namespace to support future index updates and cache invalidation.

In [5]:
import hashlib
import json
import time
import numpy as np
import redis
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [7]:

redis_client = redis.Redis(
    host="localhost",
    port=6380,
    decode_responses=True
)

print("Redis connected:", redis_client.ping())
print("Redis version:", redis_client.info("server")["redis_version"])

Redis connected: True
Redis version: 8.10.3


In [8]:
document_cache = RedisDocumentCache(
    redis_client=redis_client,
    parent_store=parent_store,
    langfuse=langfuse,
    ttl=86400
)

### 6. Test Parent Document Cache

Select an existing parent document and retrieve it twice.

**Expected behaviour:**

1. First request: Cache miss; fetch document from parent_store and save it in Redis.
2. Second request: Cache hit; retrieve document directly from Redis.

Measure both execution times and verify that the returned documents are identical.

In [9]:
parent_id = sample_parent_id

# First request: Cache miss
start = time.perf_counter()
document_1 = document_cache.get_document(parent_id)
first_latency = (time.perf_counter() - start) * 1000

# Second request: Cache hit
start = time.perf_counter()
document_2 = document_cache.get_document(parent_id)
second_latency = (time.perf_counter() - start) * 1000

print("First request:", round(first_latency, 2), "ms")
print("Second request:", round(second_latency, 2), "ms")

print("Parent ID:", parent_id)
print("Document found:", document_1 is not None)
print("Documents identical:", document_1 == document_2)

print("Cache hits:", document_cache.hits)
print("Cache misses:", document_cache.misses)

First request: 120.88 ms
Second request: 14.62 ms
Parent ID: LBezF5gB5uF80_MGGY81_OVERVIEW_001
Document found: True
Documents identical: True
Cache hits: 1
Cache misses: 1


### 7. Calculate Cache Performance

Calculate cache hit rate and miss rate to understand how effectively Redis serves parent documents.

**Hit Rate** = (Cache Hits / Total Requests) × 100

**Miss Rate** = (Cache Misses / Total Requests) × 100

We will also measure the latency improvement between the first and second requests.

In [10]:
total_requests = document_cache.hits + document_cache.misses

hit_rate = (
    document_cache.hits / total_requests * 100
    if total_requests else 0
)

miss_rate = (
    document_cache.misses / total_requests * 100
    if total_requests else 0
)

latency_reduction = (
    (first_latency - second_latency) / first_latency * 100
    if first_latency > 0 else 0
)

print(f"Cache Hit Rate: {hit_rate:.2f}%")
print(f"Cache Miss Rate: {miss_rate:.2f}%")
print(f"Latency Reduction: {latency_reduction:.2f}%")

Cache Hit Rate: 50.00%
Cache Miss Rate: 50.00%
Latency Reduction: 87.90%


### 8. Inspect Redis Document Cache Memory

Measure the memory consumed by the cached parent document.

**Purpose:**
- Understand the memory cost of document caching.
- Identify the size of individual cached records.
- Prepare for Redis memory optimization and eviction policy experiments.

In [11]:
cache_key = f"seva:document:v1:{parent_id}"

memory_bytes = redis_client.memory_usage(cache_key)

print("Document cache memory:", memory_bytes, "bytes")
print("Document cache memory:", round(memory_bytes / 1024, 2), "KB")

Document cache memory: 975 bytes
Document cache memory: 0.95 KB


### 1. Objective: LLM Response Caching

Implement an LLM response cache using Redis to avoid repeatedly invoking the LLM for identical prompts.

**Why do we need it?**

LLM inference can be expensive and time-consuming. When users submit identical questions with identical supporting context, we can reuse previously generated answers.

**Expected benefits:**
- Reduce LLM API calls.
- Reduce token consumption and API costs.
- Reduce response latency.
- Improve application throughput.

**Difference from previous caching experiments:**
- Embedding cache stores generated vectors.
- Query cache stores retrieval results.
- Document cache stores complete parent documents.
- LLM response cache stores generated answers.

**Important:** The cache key must account for the complete prompt, retrieved context, model and generation configuration. Caching only the user question can return an incorrect or outdated answer.